#### Coelostat at Kodaikanal Tower-tunnel Telescope 
3-Mirror Coelostat + Fold mirror of the polarimeter

In [12]:
# %matplotlib inline
# %matplotlib tk
%matplotlib qt5
import numpy as np
import sys
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
#
sys.path.append('P:\\Academic\\Projects\\05_HELLRIDE\\3_Development\\PyAstroPol')
import PyAstroPol as pap
pap.roundOffDisplay(5)

In [13]:
class KTTCoelostat(pap.System):
    def __init__(self, Sun, Sun_disp):
        #
        self.Sun = Sun
        Dec = np.radians(self.Sun.Declination)
        HA = np.radians(self.Sun.HourAngle)
        Lat = np.radians(self.Sun.Latitude)
        PolarAxis = np.array([0, np.sin(Lat), np.cos(Lat)])
        self.PolarAxis = PolarAxis
        #
        self.Dist_EW = 830.0  # East-West distance
        self.Dist_ZN = 740.0  # Zenth-Nadir distance
        #
        A = -np.nan_to_num(np.abs(HA)/HA, nan=-1.0)*self.Dist_EW
        B = self.Dist_ZN
        SunPos = np.array([np.sin(HA)*np.cos(Dec), 
                           np.cos(HA)*np.cos(Dec)*np.cos(Lat) + np.sin(Dec)*np.sin(Lat), 
                           -np.cos(HA)*np.cos(Dec)*np.sin(Lat) + np.sin(Dec)*np.cos(Lat)])
        #
        TEMP = np.dot(SunPos, PolarAxis)
        a = TEMP**2-PolarAxis[2]**2
        b = -2*B*PolarAxis[1]*PolarAxis[2]
        c = (A**2+B**2)*TEMP**2 - B**2*PolarAxis[1]**2
        if (a == 0):
            C = -c/b
        else:
            C = (-b + np.nan_to_num(np.abs(Dec)/Dec, nan=1.0)*np.nan_to_num(np.sqrt(b**2-4*a*c)))/2.0/a
        self.Dist_NS = np.abs(C)
        # Time-varying
        self.Center1 = np.array([-A, 0.0, -C])
        self.Center2 = np.array([0.0, B, 0.0])
        self.Incidence1 = -SunPos
        self.Reflection1 = np.array([A, B, C])/np.linalg.norm(np.array([A, B, C]))
        self.Normal1 = (self.Reflection1-self.Incidence1)/np.linalg.norm(self.Reflection1-self.Incidence1)
        self.Incidence2 = self.Reflection1
        self.Reflection2 = np.array([0.0, -1.0, 0.0])
        self.Normal2 = (self.Reflection2-self.Incidence2)/np.linalg.norm(self.Reflection2-self.Incidence2)
        # Fixed
        D = 3000       # Depth of M3 from top level
        E = 300         # Distance of fold mirror M4 from the spectrograph slit, towards east
        F = 36000      # Focus distance, towards south
        self.Center3 = np.array([0.0, -D, 0.0])
        self.Center4 = np.array([-E, -D, -F])
        self.Center5 = np.array([0, -D, -F])
        self.Reflection3 = self.Center4-self.Center3
        self.Reflection3 /= np.linalg.norm(self.Reflection3)
        self.Reflection4 = np.array([1, 0, 0])
        self.Incidence3 = self.Reflection2
        self.Incidence4 = self.Reflection3
        self.Normal3 = (self.Reflection3-self.Incidence3)/np.linalg.norm(self.Reflection3-self.Incidence3)
        self.Normal4 = (self.Reflection4-self.Incidence4)/np.linalg.norm(self.Reflection4-self.Incidence4)
        #
        self.Sun.makeOrigin(self.Center1-self.Sun.Distance*self.Incidence1)
        Sun_disp.makeOrigin(self.Center1-self.Sun.Distance*self.Incidence1)
        #
        RI_Al = 2.16-7.18j
        C12 = pap.Coating([RI_Al, 1.67], [0.100, 0.00])
        self.M1 = pap.Surface(600, Mirror=True, n2=RI_Al)
        self.M1.pointToDirection(self.Normal1)
        self.M1.makeOrigin(self.Center1)
        # self.M1.Coating = C12
        #
        self.M2 = pap.Surface(600, Mirror=True, n2=RI_Al)
        self.M2.pointToDirection(self.Normal2)
        self.M2.makeOrigin(self.Center2)
        # self.M2.Coating = C12
        #
        self.M3 = pap.Surface(600, Mirror=True, n2=RI_Al)
        self.M3.pointToDirection(self.Normal3)
        self.M3.makeOrigin(self.Center3)
        # self.M2.Coating = C12
        #
        self.M4 = pap.Surface(600, Mirror=True, n2=RI_Al)
        self.M4.pointToDirection(self.Normal4)
        self.M4.makeOrigin(self.Center4)
        # self.M2.Coating = C12
        #
        self.Window = pap.Detector(500.0)
        self.Window.rotateAboutY(90.0)
        self.Window.makeOrigin(self.Center5)
        pap.System.__init__(self, self.Sun, [self.M1, self.M2, self.M3, self.M4], self.Window, dRays=Sun_disp)
        return

In [14]:
Sun = pap.AstroSource(10000, Clear=300, Dec=6.0, HA=21.0, Lat=10.23, Dist=5000.0)       # Position of the Sun
Sun_disp = pap.AstroSource(20, Clear=300, Dec=6.0, HA=21.0, Lat=10.23, Dist=5000.0)     # Source for display
Coel = KTTCoelostat(Sun, Sun_disp)                                                      # Coelostat configuration
Coel.propagateRays() 

In [18]:
Fig = plt.figure()                                             
Ax = Fig.add_subplot(111, projection='3d')
Coel.draw(Ax)
pap.adjustAspect(Ax, 20000.0)
plt.show()

In [19]:
MM, Tra = Coel.getSystemMuellerMatrix()
print('Mueller matrix is : \n', MM)
print('Throughput is : ', Tra)

Mueller matrix is : 
 [[ 1.00000 -0.03764 -0.02103 -0.00054]
 [-0.04230  0.76150  0.64750  0.02803]
 [ 0.00837 -0.64401  0.75075  0.14087]
 [-0.00054  0.07019 -0.12531  0.98869]]
Throughput is :  0.5355423105725637
